In [1]:
import nltk

# Download the Brown corpus and the universal tagset
nltk.download("brown")
nltk.download('universal_tagset')

[nltk_data] Downloading package brown to /home/labuser/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package universal_tagset to
[nltk_data]     /home/labuser/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.


True

In [8]:
from nltk.corpus import brown

# Get the tagged sentences from the Brown corpus
tagged_sentences = brown.tagged_sents(tagset="universal")

In [10]:
universal_tags = ['ADJ', 'ADP', 'ADV', 'CONJ', 'DET', 'NOUN', 'NUM', 'PRON', 'PRT', 'VERB', '.', 'X']

In [13]:

def create_folds(data, k=5):
  fold_size = len(data) // k
  folds = []
  for i in range(k):
    start = i * fold_size
    end = (i + 1) * fold_size if i < k - 1 else len(data)
    folds.append(data[start:end])
  return folds


In [14]:
def train_hmm(training_sentences):
    transition_matrix = {}
    emission_matrix = {}
    tag_counts = {}
    word_tag_counts = {}

    for sentence in training_sentences:
        previous_tag = 'START'
        tag_counts[previous_tag] = tag_counts.get(previous_tag, 0) + 1  
        for word, tag in sentence:
            transition_matrix[previous_tag] = transition_matrix.get(previous_tag, {})
            transition_matrix[previous_tag][tag] = transition_matrix[previous_tag].get(tag, 0) + 1

            emission_matrix[tag] = emission_matrix.get(tag, {})
            emission_matrix[tag][word] = emission_matrix[tag].get(word, 0) + 1

            tag_counts[tag] = tag_counts.get(tag, 0) + 1  

            word_tag_counts[word] = word_tag_counts.get(word, {})
            word_tag_counts[word][tag] = word_tag_counts[word].get(tag, 0) + 1

            previous_tag = tag

    num_tags = len(universal_tags) + 1 
    for prev_tag in transition_matrix:
        for tag in transition_matrix[prev_tag]:
            transition_matrix[prev_tag][tag] = (transition_matrix[prev_tag][tag] + 1) / (tag_counts.get(prev_tag, 0) + num_tags)

    for tag in emission_matrix:
        for word in emission_matrix[tag]:
            emission_matrix[tag][word] = (emission_matrix[tag][word] + 1) / (tag_counts.get(tag, 0) + len(word_tag_counts))

    return transition_matrix, emission_matrix

In [15]:
folds = create_folds(tagged_sentences)

for i in range(5):
  test_data = folds[i]
  training_data = [data for j, data in enumerate(folds) if j != i]
  training_data = [sentence for fold in training_data for sentence in fold]

  transition_matrix, emission_matrix = train_hmm(training_data) 
  print(f"Fold {i+1}: Transition Matrix and Emission Matrix created.")

Fold 1: Transition Matrix and Emission Matrix created.
Fold 2: Transition Matrix and Emission Matrix created.
Fold 3: Transition Matrix and Emission Matrix created.
Fold 4: Transition Matrix and Emission Matrix created.
Fold 5: Transition Matrix and Emission Matrix created.


In [21]:
transition_matrix['START']

{'DET': 0.23245069194725945,
 '.': 0.06389887762885474,
 'PRON': 0.1391740220115506,
 'NOUN': 0.14821837201699903,
 'ADV': 0.09190367222403836,
 'ADP': 0.1364716138171516,
 'VERB': 0.04663833496785442,
 'NUM': 0.01896044458973521,
 'CONJ': 0.04864334749918274,
 'ADJ': 0.03925029966219898,
 'PRT': 0.03391086411681377,
 'X': 0.0004576659038901602}